# qbraid.runtime.pasqal

> **Authentication Notice:** This notebook authenticates **directly with Pasqal** using your Pasqal Cloud credentials. It does **NOT** use qBraid credits. You must have a valid account from the [Pasqal Cloud Portal](https://portal.pasqal.cloud) to run the examples below.

The [qBraid-SDK](https://github.com/qBraid/qBraid) provides a `PasqalProvider` that interfaces with Pasqal's neutral-atom quantum devices and emulators. Programs are expressed as [Pulser](https://pulser.readthedocs.io/) pulse sequences and submitted through Pasqal Cloud.

In [ ]:
%%capture

%pip install 'qbraid[pasqal,visualization]'

## Authentication

The `PasqalProvider` authenticates with your Pasqal Cloud account credentials. Set the following environment variables:
- `PASQAL_USERNAME`
- `PASQAL_PASSWORD`
- `PASQAL_PROJECT_ID`

Or pass them directly to the provider. Get your credentials and project ID from the [Pasqal Cloud Portal](https://portal.pasqal.cloud).

In [ ]:
import os

from qbraid.runtime.pasqal import PasqalProvider

# Requires Pasqal Cloud credentials.

username = os.getenv("PASQAL_USERNAME")  # e.g. "you@example.com"
password = os.getenv("PASQAL_PASSWORD")
project_id = os.getenv("PASQAL_PROJECT_ID")

provider = PasqalProvider(
    username=username,
    password=password,
    project_id=project_id,
)

In [ ]:
# List available Pasqal devices
devices = provider.get_devices()
for device in devices:
    print(device.id)

## Submit a Job

Pasqal devices accept Pulser `Sequence` objects as input. Here we place a single atom in a register, declare a global Rydberg channel, and add a simple $\pi$-pulse before submitting to the `EMU_FREE` emulator.

In [ ]:
import numpy as np
from pulser import Pulse, Register, Sequence
from pulser.devices import AnalogDevice
from pulser.waveforms import BlackmanWaveform

# Create a register with a single atom and build a pulse sequence
reg = Register({"q0": (0.0, 0.0)})

sequence = Sequence(reg, AnalogDevice)
sequence.declare_channel("rydberg", "rydberg_global")

pulse = Pulse.ConstantDetuning(BlackmanWaveform(1000, np.pi), 0.0, 0.0)
sequence.add(pulse, "rydberg")

sequence.draw()

In [ ]:
# Requires active Pasqal Cloud credentials.
# Submit a job to the EMU_FREE emulator.

device = provider.get_device("EMU_FREE")
job = device.run(sequence, shots=200)
result = job.result()
print(result.data.get_counts())

In [ ]:
from qbraid.visualization import plot_histogram

counts = result.data.get_counts()
plot_histogram(counts)

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>